In [1]:
import pandas as pd
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'
df = pd.read_csv(DATA_PATH)
df.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [2]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
def get_tag(tag):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'
    
def lemmatizing(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence):
    eng_stopwords = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatizing(cleaned)
    return lemmatized

In [4]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [5]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [6]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier.train(train_data)
    acc = accuracy(classifier, test_data)
    print(f"Accuracy Percentage: {acc*100:.2f}%")

    return classifier, existed_words

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [9]:
import os
import pickle

In [10]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()

        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((classifier, existed_words, vectorizer, vectors), f)
        return classifier, existed_words, vectorizer, vectors

In [11]:
def print_menu(sentence, category):
    print(f"Sentence: {sentence}")
    print(f"Category: {category}")

    print("1. Write Sentence")
    print("2. Recommend")
    print("3. NER")
    print("4. Exit")
    return

In [12]:
def write_sentence():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split()) < 3:
        sent = input("Input a sentence: ")
    return sent

In [13]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)

    if category == 1:
        return "FAKE"
    else:
        return "TRUE"

In [14]:
def load_NER(nlp, sentence):
    doc = nlp(sentence)
    dict_ents = {}

    for ent in doc.ents:
        if ent.label_ not in dict_ents.keys():
            dict_ents[ent.label_] = []
        dict_ents[ent.label_].append(ent.text)
    return dict_ents

In [15]:
def print_NER(dict_ents):
    if dict_ents:
        for key, value in dict_ents.items():
            print(f"{key}: {value}")

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [18]:
import spacy

In [19]:
def menu():
    sent = 'N/A'
    cat = 'N/A'

    classifier = None
    existed_words = None
    vectorizer = None
    vectors = None

    nlp = spacy.load("en_core_web_sm")
    ents_dicts = {}

    while True:
        print_menu(sent, cat)
        choice = input("Choice: ")
        print()
        if choice == '1':
            sent = write_sentence()

            if classifier is None or existed_words is None or vectorizer is None or vectors is None:
                classifier, existed_words, vectorizer, vectors = load_model()
            cat = classify_text(classifier, sent, existed_words)
        
        elif choice == '2':
            recommend_top_n(vectorizer, vectors, sent)
        
        elif choice == '3':
            ents_dicts = load_NER(nlp, sent)
            print_NER(ents_dicts)
        
        elif choice == '4':
            break

In [20]:
menu()

Sentence: N/A
Category: N/A
1. Write Sentence
2. Recommend
3. NER
4. Exit

Sentence: TfidfVectorizer mbg mas
Category: FAKE
1. Write Sentence
2. Recommend
3. NER
4. Exit

1. Information Systems Support Specialist  
2. Export Documentation Coordinator/Customer Service
3. URGENT Part Timers & Full Timers Required.
4. Clinic Assistant, Kingston
5. Corporate Accountant CPA
Sentence: TfidfVectorizer mbg mas
Category: FAKE
1. Write Sentence
2. Recommend
3. NER
4. Exit

ORG: ['TfidfVectorizer']
Sentence: TfidfVectorizer mbg mas
Category: FAKE
1. Write Sentence
2. Recommend
3. NER
4. Exit

